# HER Study — MoS₂ Results Analysis

This notebook post-processes VASP output files for the **Hydrogen Evolution Reaction (HER)** study on the MoS₂ monolayer.

## Workflow
1. Parse VASP output (`OUTCAR` / `vasprun.xml`) with **pymatgen** and **ASE**.
2. Compute adsorption energies $E_{\mathrm{ads}}(H^*)$.
3. Apply free-energy corrections (ZPE + entropy) to obtain $\Delta G_{H^*}$.
4. Plot the free-energy diagram.
5. Visualise the relaxed MoS₂ structure.

---
**Reference energies required** (run separate VASP jobs and fill in below):
- `E_slab`   : clean MoS₂ slab  
- `E_slab_H` : MoS₂ slab + adsorbed H  
- `E_H2`     : gas-phase H₂ molecule  

In [ ]:
# ─── Standard imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ASE — structure reading & visualisation
from ase.io import read as ase_read
from ase.visualize.plot import plot_atoms

# pymatgen — VASP output parsing
from pymatgen.io.vasp.outputs import Outcar, Vasprun
from pymatgen.core import Structure

print('All imports successful.')

## 1 · Configuration — Set Paths and Reference Energies

In [ ]:
# ─── Paths ───────────────────────────────────────────────────────────────────
OUTPUTS_DIR = Path('outputs')   # directory containing VASP output files

# Sub-directories for each calculation (create one OUTCAR / vasprun.xml per job)
slab_dir   = OUTPUTS_DIR / 'slab'        # clean MoS2 slab
slab_H_dir = OUTPUTS_DIR / 'slab_H'     # slab + H adsorbed on S-top site
H2_dir     = OUTPUTS_DIR / 'H2'         # gas-phase H2 molecule

# ─── Free-energy correction constants (literature values at 300 K) ───────────
ZPE_H_star   =  0.19   # eV  — ZPE of adsorbed H*
ZPE_H2_half  =  0.135  # eV  — ½ × ZPE of H₂
T            =  300    # K   — temperature
TS_H_star    =  0.0    # eV  — entropy of adsorbed H* (≈ 0 for strongly bound)
TS_H2_half   =  0.205  # eV  — ½ × T×S of H₂ at 300 K, 1 bar

# Net ZPE and entropy correction for H*
DELTA_ZPE = ZPE_H_star - ZPE_H2_half          # eV
DELTA_TS  = TS_H_star  - TS_H2_half           # eV  (negative value)
CORRECTION = DELTA_ZPE - DELTA_TS             # ≈ +0.24 eV (Nørskov 2005)

print(f'ΔZPE       = {DELTA_ZPE:+.3f} eV')
print(f'−TΔS       = {-DELTA_TS:+.3f} eV')
print(f'Total corr = {CORRECTION:+.3f} eV')

## 2 · Parse VASP Output Files

In [ ]:
def parse_final_energy(directory: Path, prefer_vasprun: bool = True) -> float:
    """Return the final DFT total energy (eV) from a VASP calculation directory.

    Tries vasprun.xml first (more reliable); falls back to OUTCAR.
    """
    vasprun_file = directory / 'vasprun.xml'
    outcar_file  = directory / 'OUTCAR'

    if prefer_vasprun and vasprun_file.exists():
        vr = Vasprun(str(vasprun_file), parse_dos=False, parse_eigen=False)
        energy = vr.final_energy
        print(f'  [{directory.name}] Energy from vasprun.xml : {energy:.6f} eV')
        return energy

    if outcar_file.exists():
        oc = Outcar(str(outcar_file))
        energy = oc.final_energy
        print(f'  [{directory.name}] Energy from OUTCAR      : {energy:.6f} eV')
        return energy

    raise FileNotFoundError(
        f'No vasprun.xml or OUTCAR found in {directory}.\n'
        f'Run VASP and copy output files to the outputs/ subdirectories.'
    )


print('Parsing energies...')
E_slab   = parse_final_energy(slab_dir)
E_slab_H = parse_final_energy(slab_H_dir)
E_H2     = parse_final_energy(H2_dir)

## 3 · Calculate Adsorption Energy and ΔG(H*)

In [ ]:
# ─── Adsorption energy ───────────────────────────────────────────────────────
#   E_ads(H*) = E(slab+H) - E(slab) - ½ E(H₂)
E_ads = E_slab_H - E_slab - 0.5 * E_H2

# ─── Gibbs free energy of hydrogen adsorption ────────────────────────────────
#   ΔG(H*) = E_ads + ΔZPE - TΔS
delta_G_H = E_ads + CORRECTION

print(f'\nResults')
print(f'  E_ads(H*)  = {E_ads:+.4f} eV')
print(f'  ΔG(H*)     = {delta_G_H:+.4f} eV')
print()
if abs(delta_G_H) < 0.2:
    print('  ✔  |ΔG(H*)| < 0.2 eV — MoS₂ is a promising HER catalyst.')
else:
    print('  ✘  |ΔG(H*)| ≥ 0.2 eV — adsorption is not near-thermoneutral.')

## 4 · Free-Energy Diagram

In [ ]:
# ─── Reaction coordinate data ────────────────────────────────────────────────
# Three states: ½H₂ (reference) → H* (adsorbed) → ½H₂ (desorbed)
G_states = [0.0, delta_G_H, 0.0]          # eV
labels   = ['½ H₂ (g)', 'H*', '½ H₂ (g)']
x        = [0, 1, 2]

# ─── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

# Horizontal energy levels
bar_width = 0.35
for xi, gi in zip(x, G_states):
    ax.hlines(gi, xi - bar_width / 2, xi + bar_width / 2,
              color='steelblue', linewidth=2.5)

# Connecting dashed lines
for i in range(len(x) - 1):
    ax.plot([x[i] + bar_width / 2, x[i + 1] - bar_width / 2],
            [G_states[i], G_states[i + 1]],
            'k--', linewidth=1, alpha=0.5)

# Reference line at 0
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')

# Annotations
ax.annotate(f'ΔG(H*) = {delta_G_H:+.2f} eV',
            xy=(1, delta_G_H), xytext=(1.3, delta_G_H + 0.05),
            fontsize=10, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Gibbs Free Energy (eV)', fontsize=11)
ax.set_title('HER Free-Energy Diagram — MoS₂', fontsize=12)
ax.set_xlim(-0.5, 2.5)

plt.tight_layout()
plt.savefig('HER_free_energy_diagram.png', dpi=150)
plt.show()
print('Figure saved: HER_free_energy_diagram.png')

## 5 · Structure Visualisation

In [ ]:
# ─── Read relaxed structure ───────────────────────────────────────────────────
contcar_path = slab_H_dir / 'CONTCAR'

if contcar_path.exists():
    atoms = ase_read(str(contcar_path))
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Top view (along z-axis)
    plot_atoms(atoms, axes[0], rotation=('0x,0y,0z'))
    axes[0].set_title('MoS₂ + H* — Top View', fontsize=11)
    axes[0].axis('off')
    
    # Side view (along y-axis)
    plot_atoms(atoms, axes[1], rotation=('90x,0y,0z'))
    axes[1].set_title('MoS₂ + H* — Side View', fontsize=11)
    axes[1].axis('off')
    
    # Legend
    legend_elements = [
        mpatches.Patch(color='#6666FF', label='Mo'),
        mpatches.Patch(color='#FFFF00', label='S'),
        mpatches.Patch(color='#FFFFFF', label='H', edgecolor='black'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=3, fontsize=10, frameon=False)
    
    plt.tight_layout()
    plt.savefig('MoS2_H_structure.png', dpi=150)
    plt.show()
    print('Figure saved: MoS2_H_structure.png')
else:
    print(f'CONTCAR not found at {contcar_path}.  '
          'Run VASP and copy CONTCAR to outputs/slab_H/ to visualise the structure.')

## 6 · Density of States (DOS) Analysis

In [ ]:
# ─── Plot total and projected DOS ─────────────────────────────────────────────
vasprun_file = slab_H_dir / 'vasprun.xml'

if vasprun_file.exists():
    from pymatgen.electronic_structure.plotter import DosPlotter

    vr          = Vasprun(str(vasprun_file), parse_dos=True)
    complete_dos = vr.complete_dos

    plotter = DosPlotter()
    plotter.add_dos('Total DOS', complete_dos)

    # Add element-projected DOS
    for element in complete_dos.structure.composition.elements:
        plotter.add_dos(str(element), complete_dos.get_element_dos()[element])

    fig = plotter.get_plot(xlim=(-5, 5), ylim=(-20, 20))
    fig.suptitle('Projected DOS — MoS₂ + H*', fontsize=12)
    plt.tight_layout()
    plt.savefig('MoS2_H_dos.png', dpi=150)
    plt.show()
    print('Figure saved: MoS2_H_dos.png')
else:
    print(f'vasprun.xml not found at {vasprun_file}.  '
          'Run VASP with LORBIT=11 and copy vasprun.xml to outputs/slab_H/ first.')

## 7 · Summary Table

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Quantity':  ['E(slab)', 'E(slab+H)', 'E(H₂)',
                  'E_ads(H*)', 'ΔZPE − TΔS', 'ΔG(H*)'],
    'Value (eV)': [E_slab, E_slab_H, E_H2,
                   E_ads, CORRECTION, delta_G_H],
    'Notes': [
        'DFT total energy of clean slab',
        'DFT total energy of slab + H',
        'DFT total energy of H₂ molecule',
        'Raw adsorption energy',
        'Free-energy correction (ZPE + entropy)',
        'Gibbs free energy of H adsorption',
    ],
})

print(results.to_string(index=False))